# Generics, Variance & Advanced Types



## Generics — type parameters

A *generic* method, class, or trait declares one or more **type parameters** in square brackets, and uses those parameters in its signatures. The compiler instantiates the parameters at the call site — once with `Int`, once with `String`, once with whatever — without you writing one version per type.

The shape:

```scala
def identity[A](x: A): A = x
def first[A, B](pair: (A, B)): A = pair._1
class Box[A](val value: A)
trait Container[A]:
  def get: A
```

The convention: single capital letters — `A`, `B`, `T`, `K`, `V` — for type parameters. `A` for a single payload, `K` and `V` for map keys and values, `T` from java's tradition, `R` for return when distinguishing from `T`. The names are just for readability — pick anything you like.

In [ ]:
// Generic method — type parameter inferred from the argument
def identity[A](x: A): A = x

println(identity(42))         // 42, A = Int
println(identity("scala"))    // scala, A = String
println(identity(List(1, 2))) // List(1, 2), A = List[Int]

// Generic class — type parameter fixed at construction
class Box[A](val value: A):
  def show: String = s"Box($value)"

val intBox = Box(10)
val strBox = Box("hello")
println(intBox.show)    // Box(10)
println(strBox.show)    // Box(hello)

// Generic trait — implementations specialize
trait Container[A]:
  def get: A
  def isEmpty: Boolean

class Single[A](v: A) extends Container[A]:
  def get: A = v
  def isEmpty: Boolean = false

println(Single(99).get)  // 99

## Upper and lower bounds — constraining type parameters

By default, a type parameter `A` can be replaced by any type at all. Bounds let you constrain that.

- **Upper bound — `A <: Bound`** — `A` must be a subtype of `Bound`. Inside the method, you can call any method `Bound` has.
- **Lower bound — `A >: Bound`** — `A` must be a supertype of `Bound`. Less common; mainly shows up in variance escape hatches, covered next.

Bounds are the type-level equivalent of "this only makes sense for things that can do X." Without a bound, the compiler does not know that your `A` has, say, a `.name` method.

In [ ]:
trait Named:
  def name: String

class Person(val name: String) extends Named
class Animal(val species: String, val name: String) extends Named

// Upper bound — A must be a subtype of Named, so we can call .name
def announce[A <: Named](thing: A): String =
  s"introducing: ${thing.name}"

println(announce(Person("ganesh")))
println(announce(Animal("dog", "rex")))

// Without the bound, this wouldn't compile:
//   def announce[A](thing: A): String = s"introducing: ${thing.name}"
// because the compiler doesn't know A has a .name method.

// Lower bound — A must be a supertype of Person.
// Mostly useful with variance — we'll see why in the next sections.
def widen[A >: Person](x: Person): A = x
val wide: Named = widen(Person("alice"))  // A inferred as Named
println(wide.name)

## Variance — does `List[Dog]` count as `List[Animal]`?

Imagine a `List[Dog]` and a function that takes a `List[Animal]`. Can you pass the dogs to the function?

This question is what **variance** answers. There are three possible answers, and each one is a property of the *type constructor* (`List[...]`, `Option[...]`, `Function1[..., ...]`), not of the elements:

- **Covariant** — `List[Dog]` IS a `List[Animal]`. Marked `+A` in the type parameter declaration.
- **Contravariant** — `Function[Animal]` IS a `Function[Dog]` (note the flip). Marked `-A`.
- **Invariant** — neither direction works. Default, no annotation.

The Scala standard library marks `List`, `Option`, `Try`, `Future` as covariant on their payload. `Function1` is contravariant on its parameter and covariant on its return. Mutable collections like `ArrayBuffer` are invariant.

Why does variance matter? Because the answer changes which method calls type-check. Get this wrong and you get a confusing error about a covariant or contravariant position.

In [ ]:
class Animal(val name: String)
class Dog(name: String) extends Animal(name)
class Puppy(name: String) extends Dog(name)

// Covariant container — Box[Dog] IS a Box[Animal]
class CovBox[+A](val value: A)

val dogBox: CovBox[Dog]    = CovBox(Dog("rex"))
val animBox: CovBox[Animal] = dogBox    // works! covariance
println(animBox.value.name)

// Invariant container — Box[Dog] is NOT a Box[Animal]
class InvBox[A](val value: A)
val dogBox2: InvBox[Dog] = InvBox(Dog("rex"))
// val animBox2: InvBox[Animal] = dogBox2  // would not compile

// Contravariant consumer — Printer[Animal] IS a Printer[Dog]
trait Printer[-A]:
  def print(a: A): Unit

val animalPrinter: Printer[Animal] = (a: Animal) => println(s"animal: ${a.name}")
val dogPrinter: Printer[Dog]       = animalPrinter   // works! contravariance
dogPrinter.print(Dog("rex"))

## The producer/consumer rule of thumb

The variance choice for a type parameter follows a simple rule, sometimes called **PECS** (Producers Extend, Consumers Super) from java, or the producer/consumer rule.

- If the type parameter only appears in **return positions** — methods *produce* values of type `A` — make it **covariant** (`+A`).
- If the type parameter only appears in **parameter positions** — methods *consume* values of type `A` — make it **contravariant** (`-A`).
- If it appears in both, leave it **invariant**.

The standard-library examples:

- `List[+A]` — `head: A` (produces). No method takes an `A` as input on an immutable List. → covariant.
- `Function1[-A, +B]` — `apply(a: A): B` — `A` is consumed, `B` is produced. → A contravariant, B covariant.
- `ArrayBuffer[A]` — has `apply(i): A` (produces) and `append(a: A)` (consumes). → invariant.

This rule is not just a convention — the compiler *enforces* it. If you mark a type parameter `+A` and then write a method that takes an `A` as input, the compiler rejects it with "covariant type A occurs in contravariant position.

In [ ]:
// Function1 is the canonical example
// Function1[-A, +B] means:
//   - You can use a Function1[Animal, Dog] wherever a Function1[Dog, Animal]
//     is expected — input narrowed, output widened.

val animalToDog: Function1[Animal, Dog] = (a: Animal) => Dog(a.name)

// Substitutable: Function1[Dog, Animal] expected, but we have Function1[Animal, Dog]
val dogToAnimal: Function1[Dog, Animal] = animalToDog

println(dogToAnimal(Dog("rex")).name)

// Why does this work?
// - On input: any Dog IS an Animal, so any caller passing a Dog can be served
//   by a function that accepts the wider Animal.
// - On output: any Dog IS an Animal, so any caller expecting an Animal back
//   is happy with a Dog.

## Lower bounds on covariant type parameters

A method with a covariant type parameter cannot take that parameter as an input — that would put a covariant `A` in a contravariant position. But sometimes you legitimately want to. The standard workaround is to introduce a **fresh** type parameter with a lower bound:

```scala
class CovList[+A]:
  def prepend[B >: A](x: B): CovList[B] = ???
```

The new parameter `B` is a supertype of `A` — when you prepend a `B` to a list of `A`s, you get a list of `B`s. This is exactly what immutable `List#::` does in the standard library.

In [ ]:
// Simplified immutable list with the same variance pattern as scala.List
sealed trait MyList[+A]:
  def prepend[B >: A](x: B): MyList[B] = Cons(x, this)

case class Cons[+A](head: A, tail: MyList[A]) extends MyList[A]
case object MyNil                              extends MyList[Nothing]

class Animal(val name: String)
class Dog(name: String) extends Animal(name)

val dogs: MyList[Dog] = Cons(Dog("rex"), Cons(Dog("buddy"), MyNil))

// Prepending an Animal to a list of Dog gives a list of Animal
val animals: MyList[Animal] = dogs.prepend(Animal("generic"))

// Inspect — every element is at least an Animal
def names(xs: MyList[Animal]): List[String] = xs match
  case Cons(a, rest) => a.name :: names(rest)
  case MyNil         => Nil

println(names(animals))

## Type erasure — what the JVM forgets at runtime

Scala generics, like java generics, are **erased** at compile time. The java virtual machine sees `List` at runtime, not `List[Int]`. The type parameter is purely a compile-time check.

The practical consequence: a pattern match cannot distinguish `List[Int]` from `List[String]` at runtime. The compiler warns you when you try.

The escape hatch is `TypeTag` / `ClassTag` — runtime tokens that carry type information past erasure. Spark uses `ClassTag` extensively for serialization, which is why so many Spark methods carry an implicit `ClassTag` parameter.

In [ ]:
import scala.reflect.ClassTag

// The warning is real — uncomment and you'll see it at compile time:
//   the type test for List[Int] cannot be checked at runtime
// def isListOfInt(x: Any): Boolean = x match
//   case _: List[Int]    => true
//   case _               => false

// What you can check: the outer type (List), not the inner parameter
def isList(x: Any): Boolean = x match
  case _: List[?] => true
  case _          => false

println(isList(List(1, 2)))      // true
println(isList(List("a", "b")))  // true — same answer
println(isList("not a list"))    // false

// ClassTag — carries enough runtime info to construct typed arrays
def newArrayOf[A: ClassTag](size: Int, default: A): Array[A] =
  val arr = new Array[A](size)
  for i <- 0 until size do arr(i) = default
  arr

println(newArrayOf[Int](3, 99).mkString(", "))         // 99, 99, 99
println(newArrayOf[String](2, "hi").mkString(", "))   // hi, hi

## Opaque types — zero-cost nominal wrappers

A type alias (notebook 02) does not create a new type — `type UserId = String` and `String` are interchangeable. A `class UserId(val raw: String)` is a different type but allocates a wrapper object per instance.

**Opaque types** are Scala 3's middle ground. They are a new nominal type at compile time — the compiler refuses to mix them with the underlying — but at runtime they are exactly the underlying type, with no boxing.

The shape: declare with `opaque type` inside an object. Outside that object, the type is opaque (you cannot see through it). Inside the object, you can convert freely. Define a constructor (often `apply`) and any allowed operations as members of the object.

In [ ]:
object Ids:
  opaque type UserId    = String
  opaque type AccountId = String

  // Constructors and operations live inside, where we can see through
  object UserId:
    def apply(raw: String): UserId = raw

  object AccountId:
    def apply(raw: String): AccountId = raw

  extension (id: UserId)    def raw: String = id
  extension (id: AccountId) def raw: String = id

import Ids.*

val u = UserId("u-001")
val a = AccountId("a-042")

// Outside Ids, these are distinct types — the compiler enforces it
def processUser(id: UserId): String = s"user: ${id.raw}"
println(processUser(u))
// println(processUser(a))    // would not compile — UserId required, got AccountId

// At runtime, both are just String — zero overhead
println(u.isInstanceOf[String])  // true

## Intersection types — `A & B`

`A & B` is the type of values that are *both* an `A` and a `B`. Replaces the older `A with B` in type position. Useful for combining the surface of two traits without naming the combination.

In [ ]:
trait Named:
  def name: String

trait Aged:
  def age: Int

// Intersection — has both surfaces
def describe(person: Named & Aged): String =
  s"${person.name}, ${person.age}"

class Member(val name: String, val age: Int) extends Named, Aged
println(describe(Member("ganesh", 30)))

## Union types — `A | B`

`A | B` is the type of values that are *either* an `A` or a `B`. Replaces a common Scala 2 trick where you had to invent a `sealed trait` or use `Either` to express "one of these two unrelated types."

You destructure a union type by pattern matching.

In [ ]:
// A field that holds an Int or a String — without an Either or a wrapping trait
def stringify(x: Int | String): String = x match
  case n: Int    => s"int $n"
  case s: String => s"str $s"

println(stringify(42))
println(stringify("hello"))

// Composable — works in collections, in function returns, anywhere
val mixed: List[Int | String] = List(1, "two", 3, "four")
val described = mixed.map(stringify)
println(described)

## Match types — types that compute

A **match type** is a type-level conditional. Given an input type, it computes an output type by matching on the shape of the input. The shape mirrors a value-level `match`, but at the type level.

This is the deepest part of the type system you are likely to read about in everyday Scala. The standard library uses it in `Tuple` operations and `IArray`. You almost certainly do not need to write one — but recognising the shape lets you read Scala 3 library signatures.

In [ ]:
// A type-level conditional: Element[T] is the element type of T
type Element[T] = T match
  case String      => Char
  case Array[t]    => t
  case Iterable[t] => t

// At the type level, the compiler computes:
//   Element[String]       =:= Char
//   Element[Array[Int]]   =:= Int
//   Element[List[String]] =:= String

// You can prove it works with a typed val
val a: Element[String]       = 'x'    // Char
val b: Element[Array[Int]]   = 42     // Int
val c: Element[List[String]] = "hi"   // String

println(s"$a $b $c")

## Structural types — duck typing at the type level

A structural type describes the *shape* of an object — what methods it must have — without naming a class or trait. The compiler then accepts any value with that shape.

In Scala 3, structural types require `import scala.language.reflectiveCalls` (or the `selectDynamic` mechanism) and use reflection at the call site, which is slow. Use them sparingly. Mentioned here only so the syntax does not surprise you.

```scala
def closeIt(resource: { def close(): Unit }): Unit =
  resource.close()
```

Reach for a trait + extension or a type class before reaching for structural types.

## Self types — `this: X =>`

A self type declares "this trait requires that any class mixing it in must also be an `X`." Different from `extends X` because the trait does not actually inherit `X` — it just demands the implementation provides one.

The classic use is the *cake pattern*, where dependencies between modular components are expressed as self types. You will see it in older Scala codebases and in some library internals. In Spark, `SQLContext` uses self types to refer to `SparkContext`.

In [ ]:
trait Database:
  def query(sql: String): List[String]

trait UserService:
  this: Database =>   // self type: any class mixing in UserService must also be a Database
  def findUser(id: Int): List[String] =
    query(s"SELECT * FROM users WHERE id = $id")

class App extends Database, UserService:
  def query(sql: String): List[String] = List(s"<results for: $sql>")

println(App().findUser(42))

## Path-dependent types — briefly

Types can be members of values, not just members of classes. The resulting type is a *path-dependent type* — its full name includes the value it lives on.

```scala
class Outer:
  class Inner
val a = Outer()
val b = Outer()
val x: a.Inner = a.Inner()    // type a.Inner — bound to value a
// val y: b.Inner = a.Inner()  // would not compile — a.Inner and b.Inner are distinct
```

This is exotic but real — Spark's `Dataset[T]` exploits it in places. Mentioned for recognition; you will almost never write a path-dependent type yourself.

## Reading harder Scala signatures

The standard-library signature for `Future.flatMap` looks like this:

```scala
def flatMap[S](f: T => Future[S])(using ExecutionContext): Future[S]
```

Decoded:

- `[S]` — declares a fresh type parameter `S`, the result type of the chained future.
- `(f: T => Future[S])` — a function from the original payload type `T` to a future of `S`.
- `(using ExecutionContext)` — a second parameter list, providing the execution context implicitly. Notebook 08 covers `using`.
- `: Future[S]` — the return.

A more dramatic example from `cats.Traverse`:

```scala
def traverse[G[_]: Applicative, A, B](fa: F[A])(f: A => G[B]): G[F[B]]
```

`G[_]` is a higher-kinded type parameter — `G` is itself a type constructor (like `List` or `Option`). `: Applicative` is a context bound — `G` must have an `Applicative` instance available. We will meet context bounds and type classes in notebook 08.

You do not need to write this kind of signature in everyday code. But you will read it, and the rules in this notebook are how you decode it.